# Last.fm album recommendations (interactive)

Similar-album recommendations via Last.fm's `track.getSimilar` API.

Compare three ways of picking the seed track from the album:

- **(a) Top listener** — the track with the most Last.fm listeners on the album
- **(b) Random** — a random track from the album tracklist
- **(c) Top-N tracks** — use the `top_n` tracks (default 3) with the most listeners, aggregate recommendations by vote count; falls back to **(a)** if nothing is found

Run one strategy cell at a time, then compare overlap at the bottom.

Set `LASTFM_API_KEY` in `lastfm/.env (or the repo-root .env)` (see `.env.example`).


In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from lastfm.recommender import get_album_info, recommend_album


## Configuration


In [ ]:
N_RECS = 5           # recommendations per strategy
FETCH_FLOOR = 10     # similar tracks fetched per seed
TOP_N = 3            # seed tracks for top_n_tracks
RANDOM_SEED = 42     # random track selection


## Seed album


In [ ]:
SEED_ALBUM = "Take Care"
SEED_ARTIST = "Drake"

seed = get_album_info(SEED_ARTIST, SEED_ALBUM)
pd.Series({k: seed[k] for k in seed if k != "tracks"})


## Seed-track strategies

Each cell calls `recommend_album()` directly. Use `clear_cache=False` after the first run to avoid redundant API calls.


### (a) Top-listener track


In [ ]:
_, seed_track_a, recs_a, _ = recommend_album(
    SEED_ALBUM,
    artist=SEED_ARTIST,
    n_recs=N_RECS,
    fetch_floor=FETCH_FLOOR,
    track_selection="top_listener",
)
recs_a


### (b) Random track


In [ ]:
_, seed_track_b, recs_b, _ = recommend_album(
    SEED_ALBUM,
    artist=SEED_ARTIST,
    n_recs=N_RECS,
    fetch_floor=FETCH_FLOOR,
    track_selection="random",
    random_seed=RANDOM_SEED,
    clear_cache=False,
)
recs_b[["artist", "album", "matched_via"]]


### (c) Top-N tracks by listeners


In [ ]:
_, seed_track_c, recs_c, used_fallback_c = recommend_album(
    SEED_ALBUM,
    artist=SEED_ARTIST,
    n_recs=N_RECS,
    fetch_floor=FETCH_FLOOR,
    track_selection="top_n_tracks",
    top_n=TOP_N,
    clear_cache=False,
)
print(f"Used fallback? {used_fallback_c}")
recs_c[["artist", "album", "seed_track", "matched_via", "similarity_score"]]


## Compare strategies


In [ ]:
# Overlap between (a) and (b) on album key
recs_a.merge(recs_b, on="key", suffixes=("_top_listener", "_random"))
